## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
import httpx
import logging

logger = logging.getLogger(__name__)


In [2]:
load_dotenv(override=True)

True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [3]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 500 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="medium")],
    model="gpt-4.1-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [4]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2025, several AI agent frameworks have emerged, each offering unique capabilities for developing intelligent systems:

- **OpenAI Agents SDK**: A lightweight Python framework released in March 2025, focusing on creating multi-agent workflows with comprehensive tracing and guardrails. It is compatible with over 100 different large language models (LLMs). ([jlcnews.com](https://www.jlcnews.com/post/the-best-ai-agents-in-2025-tools-frameworks-and-platforms-compared?utm_source=openai))

- **Google Agent Development Kit (ADK)**: Announced in April 2025, this modular framework integrates with Google's ecosystem, including Gemini and Vertex AI. It supports hierarchical agent compositions and requires minimal coding, facilitating efficient development. ([jlcnews.com](https://www.jlcnews.com/post/the-best-ai-agents-in-2025-tools-frameworks-and-platforms-compared?utm_source=openai))

- **Agent Lightning**: Introduced in August 2025, Agent Lightning is a flexible and extensible framework that enables reinforcement learning-based training of LLMs for any AI agent. It achieves decoupling between agent execution and training, allowing seamless integration with existing agents developed through various methods. ([arxiv.org](https://arxiv.org/abs/2508.03680?utm_source=openai))

- **Gradientsys**: A next-generation multi-agent scheduling framework that coordinates diverse specialized AI agents using a typed Model-Context Protocol (MCP) and a ReAct-based dynamic planning loop. It supports hybrid synchronous/asynchronous execution and incorporates a robust retry-and-replan mechanism to handle failures gracefully. ([arxiv.org](https://arxiv.org/abs/2507.06520?utm_source=openai))

- **Cognitive Kernel-Pro**: An open-source framework designed to democratize the development and evaluation of advanced AI agents. It focuses on curating high-quality training data for Agent Foundation Models and integrates advanced features, including multi-turn contextual dialogue and dynamic safety validation. ([arxiv.org](https://arxiv.org/abs/2508.00414?utm_source=openai))

- **OpenClaw**: An open-source autonomous AI assistant software that can execute tasks via LLMs, using messaging platforms as its main user interface. It gained popularity in late January 2026 and is known for its open-source nature. ([en.wikipedia.org](https://en.wikipedia.org/wiki/OpenClaw?utm_source=openai))

- **Agent2Agent (A2A)**: An open protocol announced by Google in April 2025, defining how AI agents communicate with each other across different systems. It allows agents built by different vendors or frameworks to discover one another, exchange messages, and coordinate tasks. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Agent2Agent?utm_source=openai))

These frameworks represent the forefront of AI agent development in 2025, each contributing to the advancement of intelligent, autonomous systems across various applications. 

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [5]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 15 # 5 keep this number low to save costs

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4.1-mini",
    output_type=WebSearchPlan,
)

In [6]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search Planner"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find the most up-to-date AI agent frameworks released or popular in 2025', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To explore comprehensive lists or comparisons of AI agent frameworks as of 2025', query='best AI agent frameworks comparison 2025'), WebSearchItem(reason='To identify new trends and cutting-edge developments in AI agent frameworks in 2025', query='AI agent frameworks trends 2025'), WebSearchItem(reason='To find notable AI research papers introducing new AI agent architectures in 2025', query='2025 AI agent framework research papers'), WebSearchItem(reason='To check major AI companies releasing AI agent frameworks in 2025', query='AI companies AI agent frameworks 2025'), WebSearchItem(reason='To gather announcements and roadmaps for AI agent platforms in 2025', query='AI agent platform announcements 2025'), WebSearchItem(reason='To look for popular open-source AI agent frameworks updated or launched in 2025', query=

In [7]:
# ── Email configuration ────────────────────────────────────────────

RESEND_API_KEY = os.environ.get("RESEND_API_KEY", "")
MAIL_FROM = os.environ.get("MAIL_FROM", "no-reply@info.patrickcmd.dev")  # Change to your verified sender domain
MAIL_TO = os.environ.get("MAIL_TO", "example@example.com")        # Change to your recipient
SENDER_NAME = "AI Sales Agent"
RESEND_API_URL = "https://api.resend.com/emails"
RESEND_REQUEST_TIMEOUT = 10.0


async def _send_via_resend(
    *,
    to: str,
    subject: str,
    text: str | None = None,
    html: str | None = None,
) -> dict:
    """Low-level async helper that posts to the Resend HTTP API."""
    payload: dict = {
        "from": f"{SENDER_NAME} <{MAIL_FROM}>",
        "to": [to],
        "subject": subject,
    }
    if html:
        payload["html"] = html
    if text:
        payload["text"] = text

    async with httpx.AsyncClient(timeout=RESEND_REQUEST_TIMEOUT) as client:
        response = await client.post(
            RESEND_API_URL,
            json=payload,
            headers={"Authorization": f"Bearer {RESEND_API_KEY}"},
        )

        if response.status_code == 401:
            logger.error("Resend auth failed — check RESEND_API_KEY")
            raise RuntimeError("Resend authentication failed")
        if response.status_code == 422:
            logger.error(f"Resend rejected request: {response.text}")
            raise RuntimeError(f"Resend validation error: {response.text}")
        if response.status_code == 429:
            logger.error("Resend rate limit hit (free tier: 100/day, 3000/month)")
            raise RuntimeError("Resend rate limit exceeded")

        response.raise_for_status()

    data = response.json()
    logger.info(f"Email sent to {to} — id: {data.get('id')}")
    return data

In [8]:
@function_tool
async def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email given the subject and HTML body """
    try:
        result = await _send_via_resend(
            to=MAIL_TO,
            subject=subject,
            html=html_body,
        )
        return {"status": "success", "id": result.get("id")}
    except Exception as e:
        logger.exception("Failed to send HTML email")
        return {"status": "error", "detail": str(e)}

In [9]:
send_email

FunctionTool(name='send_email', description='Send out an email given the subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x12fa376a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [10]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4.1-mini",
)



In [11]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4.1-mini",
    output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [12]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [13]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def deliver_report_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [14]:
query ="Latest AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await deliver_report_email(report)  
    print("Hooray!")




Starting research...
Planning searches...
Will perform 15 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


In [16]:
async def research_agent(query: str):
    """ Use the research agent to research a query """
    with trace("Research trace"):
        print("Researching...")
        search_plan = await plan_searches(query)
        search_results = await perform_searches(search_plan)
        report = await write_report(query, search_results)
        await deliver_report_email(report)  
        print("Hooray!")

In [17]:
query = "AI and LLM Engineer Roadmap in 2026."
await research_agent(query)

Researching...
Planning searches...
Will perform 15 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


In [ ]:
query = "AI and LLM Engineering Roadmap in 2026. What are the latest trends and best practices?"
await research_agent(query)

In [ ]:
query ="Latest AI Agent frameworks in 2026"
await research_agent(query)

In [ ]:
query = "I want to learn deploying django python backend applications with docker and kubernetes to AWS in production. What are the best practices and tools for this?"
await research_agent(query)

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>